# **El modelo del espacio vectorial**

# 1️⃣ Bag of Words y N-Grams


En recuperación de información, una unidad de texto se llama típicamente _documento_, aunque un documento puede ser cualquier cosa, desde un mensaje de texto hasta una novela completa. A una colección de documentos se le llama _corpus_.
En esta lección, trabajaremos con un corpus de Jorge Luis Borges extraídos de https://www.literatura.us/borges/.

Por tanto lo primero que tenéis que hacer es crear la carpeta `borges_dir` y subir ahí los ficheros `.txt`



## Bag of words

In [ ]:
from pathlib import Path
import pandas as pd
import os

currentDirectory = Path().resolve()
borges_dir = os.path.join(currentDirectory, "borges_dir")
borges_files = [
    "el_zahir.txt",
    "evangelio_segun_marcos.txt",
    "funes_el_memorioso.txt",
    "la_biblioteca_de_babel.txt",
    "deutsches_requiem.txt",
    "la_loteria_en_babilonia.txt",
]

# Crear Serie de pandas
docs_borges = pd.Series(dtype=str)
for file in borges_files:
    file_path = os.path.join(borges_dir, file)
    with open(file_path, "r", encoding="utf-8") as f:
        docs_borges[file[:-4]] = f.read()

docs_borges

Supongamos, por ejemplo, que queremos determinar cuáles son los cuentos escritos por Borges más similares o agrupar los cuentos en varios tipos. Para ello, tenemos que convertir estos documentos a forma tabular. En primer lugar nos centraremos en una representación en forma de _bolsa de palabras_ (_bag of words_).

Como ya comentarmos, la representación en forma de _bag of words_ reduce un documento únicamente al multiconjunto de sus palabras, ignorando la gramática y el orden de las palabras. (Un _multiconjunto_ es como un conjunto, excepto que los elementos pueden aparecer más de una vez).

Por ejemplo, la representación de **bag of words** de:
"_El hecho sucedió en la estancia Los Álamos, en el partido de Junín, hacia el sur, en los últimos días del mes de marzo de 1928. Su protagonista fue un estudiante de medicina, Baltasar Espinosa._" (las dos primeras líneas de _Evangelio según Marcos_) sería: `{El, hecho, sucedió, en, la, estancia, Los, Álamos, en, el, partido, de, Junín, hacia, el, sur, en, los, últimos, días, del, mes, de, marzo, de, 1928, Su, protagonista, fue, un, estudiante, de, medicina, Baltasar, Espinosa}`. En Python, es más fácil representar multiconjuntos usando diccionarios, donde las claves son las palabras únicas y los valores son sus conteos. Así, representaríamos la bolsa de palabras anterior como:
`{'El': 1,
  'hecho': 1,
  'sucedió': 1,
  'en': 3,
  'la': 1,
  'estancia': 1,
  'Los': 1,
  'Álamos': 1,
  'el': 2,
  'partido': 1,
  'de': 4,
  'Junín': 1,
  'hacia': 1,
  'sur': 1,
  'los': 1,
  'últimos': 1,
  'días': 1,
  'del': 1,
  'mes': 1,
  'marzo': 1,
  '1928': 1,
  'Su': 1,
  'protagonista': 1,
  'fue': 1,
  'un': 1,
  'estudiante': 1,
  'medicina': 1,
  'Baltasar': 1,
  'Espinosa': 1}`.

Ahora vamos a convertir los capítulos de Borges a una representación de _bag of words_. Para esto, usaremos el objeto `Counter` del módulo `collections` de la biblioteca estándar de Python. Primero, vamos a usar `Counter`.

In [ ]:
from collections import Counter

Counter(
    [
        "El",
        "hecho",
        "sucedió",
        "en",
        "la",
        "estancia",
        "Los",
        "Álamos",
        "en",
        "el",
        "partido",
        "de",
        "Junín",
        "hacia",
        "el",
        "sur",
        "en",
        "los",
        "últimos",
        "días",
        "del",
        "mes",
        "de",
        "marzo",
        "de",
        "1928",
        "Su",
        "protagonista",
        "fue",
        "un",
        "estudiante",
        "de",
        "medicina",
        "Baltasar",
        "Espinosa",
    ]
)

`Counter` recibe una lista y devuelve un diccionario con los conteos; en otras palabras, la representación de _bag of words_ que queremos.

Pero para poder usar `Counter`, primero debemos convertir nuestro documento en una lista de palabras.
Podemos hacer esto utilizando los métodos de cadenas en Pandas, como `.str.split()`, que divide una cadena en una lista basándose en algún carácter (que, por defecto, es un espacio en blanco).

Se ignoran las dos primeras líneas de cada documento, el cual indica el título del cuento y su autor, antes de convertirlo en lista de palabras



In [ ]:
docs_borges.str.split("\n").apply(lambda lines: " ".join(lines[2:]).split())

---


⚠️ Por cuestiones de simplicidad, no vamos a realizar un preprocesado tan exhaustivo como se hizo en la práctica 4.


---

Hay varios problemas con este enfoque:

* **Sensibilidad a mayúsculas y minúsculas:**
Las palabras "_El_" y "_el_" son técnicamente cadenas diferentes y serán tratadas como palabras distintas por `Counter`.

* **Puntuación:**
Por ejemplo, en _el_zahir_, las palabras "Zahir," y "Zahir." se tratarán como palabras separadas.

Podemos **normalizar** el texto para resolver estos problemas de la siguiente manera:

* Convertir todos los caracteres a minúsculas usando el método `.str.lower()`.

* Eliminar la puntuación usando una expresión regular, como por ejemplo, `[^\w\s]` indica a Python que busque cualquier patrón que no `(^)` sea un carácter alfanumérico `(\w)` o un espacio en blanco `(\s)`.

Esto detecta cualquier ocurrencia de puntuación. Luego usamos `.str.replace()` para reemplazar todas las ocurrencias detectadas por un espacio en blanco, eliminando efectivamente toda la puntuación de la cadena.

Al encadenar estos comandos, obtenemos una lista a la que podemos aplicar `Counter` para obtener la representación de _bag of words_.

In [ ]:
bag_words = docs_borges.str.lower().str.replace(r"[^\w\s]", " ", regex=True).str.split()

bag_words

In [ ]:
bag_words.apply(Counter)

Puesto que los índices son los nombres de los archivo sin `.txt`:

In [ ]:
words_bow = bag_words.apply(Counter)

print(words_bow["evangelio_segun_marcos"].most_common(10))

Este resultado muestra las 10 palabras más frecuentes del cuento **Evangelio según Marcos**.

## N-Grams

El problema de la representación de _bag of words_ es que se pierde el orden de las palabras. Por ejemplo, las siguientes oraciones tienen exactamente la misma representación de bolsa de palabras, pero transmiten significados diferentes:

1. El perro mordió a su dueño.

2. Su perro mordió al dueño.

La primera oración tiene solo dos actores (el perro y su dueño), pero la segunda oración tiene tres (una mujer, su perro y el dueño de algo). Para capturar mejor el significado _semántico_ de estos dos documentos, podemos usar **bigramas** en lugar de palabras individuales.
Un bigram es simplemente un par de palabras consecutivas. La "bolsa de **bigramas**" de las dos oraciones anteriores es bastante diferente:

1. {"El perro", "perro mordió", "mordió a", "a su", "su dueño"}

2. {"Su perro", "perro mordió", "mordió al", "al dueño"}

Solo comparten 1 bigrama en común, a pesar de compartir las mismas palabras.

Ahora vamos a obtener la representación de bigramas para las palabras anteriores. Para generar los bigramas a partir de la lista de palabras, usaremos la función `zip` en Python, que toma dos listas y devuelve una lista de pares (cada par consiste en un elemento de cada lista).

In [ ]:
list(zip([1, 2, 3], [4, 5, 6]))

In [ ]:
def get_bigrams(words):
    # Necesitamos alinear las palabras de la siguiente manera:
    #   words[0], words[1]
    #   words[1], words[2]
    #       ... ,  ...
    # words[n-1], words[n]
    #   words[n]
    # La primera lista es más larga, por lo que el último elemento de la primera lista se ignora.
    return zip(words, words[1:])


bag_words.apply(get_bigrams).apply(Counter)

En lugar de tomar 2 palabras a la vez, podríamos tomar 3, 4 o, en general, _n_ palabras. Una tupla de _n_ palabras consecutivas se llama un n-grama, y podemos convertir cualquier documento en una representación de "bolsa de n-gramas".

Cuanto mayor sea _n_, mejor capturará la representación el significado de un documento. Pero si _n_ es tan grande que casi ningún n-grama ocurre más de una vez, entonces no aprenderemos mucho de esta representación.


---

En RI tradicional (motores como Lucene, Whoosh): Normalmente se usan unigramas para la mayoría de consultas.

* Bigramas o trigramas se usan solo para frases específicas o autocorrección.

* N-gramas muy largos se usan más en modelos de lenguaje y NLP, no tanto en motores de búsqueda clásicos.


## **Ejercicio**

Encuentra la representación de trigramas para los cuentos de Borges proporcionados.

¿Cómo podrías usar los trigramas para determinar qué cuento es el más diverso lingüísticamente? ¿Cuál libro sería?

# 2️⃣ El modelo vectorial

En la sección anterior, hemos visto como un simple documento puede convertirse a una representación _bag of words_ (o, también a una representación de $n$-gramas). Ahora, en esta sección damos un paso más, y convertiremos todo el corpus de documentos en datos tabulares.

## Frecuencias de términos

La representación de bolsa de palabras nos da un mapeo entre palabras y sus conteos, como `{..., "evangelio": 4, "según": 1, "marcos": 3, ...}`.

Para convertir la bolsa de palabras en un vector de números, simplemente podemos tomar los conteos de palabras, de la siguiente manera:

| ... | evangelio  | según | marcos | ... |
|-----|------------|-------|--------|-----|
| ... |      4     |   2   |    3   | ... |

Si hacemos esto para cada documento del corpus y apilamos las filas, obtenemos una tabla de números llamada **matriz de frecuencia de términos**.

|        | evangelio  | según | marcos | ... |
|--------|------------|-------|--------|-----|
|**el_zahir**| 0 | 5 |  0 | ... |
|**evangelio_segun_marcos**|      4     |   2   |    3   | ... |
|**funes_el_memorioso**| 0 | 0 | 0 | ... |
|**la_biblioteca_de_babel**| 3 | 0 | 0 | ... |
|**deutsches_requiem**| 0 | 0 | 0 | ... |
|**la_loteria_en_babilonia**| 0 | 1 | 0 | ... |

Las columnas son todas las palabras (o términos) que aparecen en el corpus, que colectivamente forman el vocabulario.
La idea de representar documentos mediante un vector de números se llama **modelo de espacio vectorial**.

In [ ]:
import matplotlib.pyplot as plt

word = "fue"
counts = words_bow.apply(lambda counter: counter.get(word, 0))
print(counts)

counts.plot(
    kind="bar", figsize=(10, 5), title=f'Conteo de la palabra "{word}" por documento'
)
plt.ylabel("Frecuencia")
plt.show()

Vamos a obtener la matriz de frecuencia de términos para los cuentos. En la primera sección ya hemos leído los datos y le hemos aplicado la representación de _bag of words_ al texto normalizado (en minúsculas, los carateres que no son alfanuméricos los hemos sustituido por espacios, etc...)

Para convertir esto en una matriz de frecuencia de términos, necesitamos crear un `DataFrame`, donde cada columna represente una palabra y cada fila un documento, y cada celda contenga el conteo de esa palabra en el documento.


In [ ]:
bag_of_words = (
    docs_borges.str.lower()  # convert all letters to lowercase
    .str.replace(
        r"[^\w\s]", " ", regex=True
    )  # replace non-alphanumeric characters by whitespace
    .str.split("\n")
    .apply(lambda lines: " ".join(lines[2:]).split())  # split on whitespace
).apply(Counter)

bag_of_words

In [ ]:
tf = pd.DataFrame(list(bag_of_words))
tf

Esta matriz está llena de *valores faltantes*. Un valor faltante significa que la palabra no apareció en ese documento.
En otras palabras, un conteo `NaN` realmente significa un conteo de 0. Por lo tanto, tiene sentido en esta situación reemplazar los `NaN` por 0.

In [ ]:
tf = tf.fillna(0)
tf

### Implementación usando scikit-learn

También podríamos haber usado el `CountVectorizer` de `scikit-learn` para obtener la matriz de frecuencia de términos. Este vectorizador se ajusta a una lista de documentos del corpus.

Por defecto, convierte todas las letras a minúsculas y elimina la puntuación, aunque este comportamiento se puede personalizar usando los parámetros `strip_accents=` y `lowercase=`.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(lowercase=True)
# Eliminar las dos primeras líneas de cada documento
docs_borges_noheader = docs_borges.apply(lambda text: "\n".join(text.splitlines()[2:]))
vec.fit(docs_borges_noheader)  # This determines the vocabulary.
tf_sparse = vec.transform(docs_borges_noheader)

tf_sparse

Observa que `CountVectorizer` devuelve la matriz de frecuencia de términos, pero no como un `DataFrame` ni siquiera como un array de `numpy`, sino como una matriz dispersa (_sparse matrix_) de `scipy`. Una matriz dispersa es aquella cuyos elementos son mayormente ceros. Por ejemplo:

$$ \begin{pmatrix} 0 & 1.1 & 0 & 0 & 5.1 \\ 4.7 & 0 & 0 & 0 & 0 \\ 0 & 0 & 3.6 & 0 & 0 \\ 0 & 0 & 0 & 1.8 & 0 \end{pmatrix} $$

es un ejemplo de matriz dispersa.

En lugar de almacenar los 20 valores (la mayoría iguales a 0), podemos almacenar solo las posiciones de los elementos distintos de cero y sus valores:

* $(0, 1) \rightarrow 1.1$
* $(0, 4) \rightarrow 5.1$
* $(1, 0) \rightarrow 4.7$
* $(2, 2) \rightarrow 3.6$
* $(3, 3) \rightarrow 1.8$

Todos los demás elementos de la matriz se asumen como cero.
Esta representación ahorra mucha memoria cuando hay pocos elementos distintos de cero. (Pero si hay muchos, esta representación puede ser más costosa). Las matrices de frecuencia de términos suelen ser dispersas porque la mayoría de las palabras no aparecen en todos los documentos.

El formato de matriz dispersa de `scipy` se utiliza para almacenar estas matrices. Si es necesario, una matriz dispersa de `scipy` se puede convertir en una matriz de `numpy` usando la función `.todense()`.

In [ ]:
tf_sparse.todense()

Podemos convertir esta matriz de `numpy` a un `DataFrame` de `pandas`. Para que los nombres de las columnas sean descriptivos, usamos el método `.get_feature_names()` del `CountVectorizer`, que devuelve una lista de las palabras en el orden en que aparecen en la matriz.

In [ ]:
import pandas as pd

df_tf = pd.DataFrame(tf_sparse.todense(), columns=vec.get_feature_names_out())

df_tf

La matriz de frecuencia de términos que produjo `CountVectorizer` no es exactamente la misma que la que generamos manualmente usando solo `pandas`. Aunque ambas matrices tienen el mismo número de filas (6, correspondiente al número de documentos en el corpus), tienen un número diferente de columnas.

Parece que `CountVectorizer` tiene un vocabulario 13 palabras más pequeño (4267 palabras en lugar de 4280).

Podemos determinar exactamente cuáles son estas 13 palabras tomando la diferencia de conjuntos:

In [ ]:
print(len(set(tf.columns)))

print(len(set(vec.get_feature_names_out())))

print(len(set(tf.columns) - set(vec.get_feature_names_out())))

Esto nos mostrará las palabras que aparecen en nuestra matriz manual pero que `CountVectorizer` no incluyó en su vocabulario.

In [ ]:
# vocabulario de CountVectorizer
vocab_vec = set(vec.get_feature_names_out())

# vocabulario manual (por ejemplo, obtenido de pandas)
vocab_manual = set(tf.columns)

# palabras que faltan en CountVectorizer
missing_words = vocab_manual - vocab_vec
print(missing_words)

Vemos que todas las palabras que `CountVectorizer` omitió tenían solo un carácter. Por defecto, `CountVectorizer` solo conserva palabras que tengan al menos 2 caracteres.

Este comportamiento se puede personalizar usando el parámetro `token_pattern=`, pero no lo modificaremos aquí, ya que las palabras de una sola letra normalmente no son útiles para el análisis.

`CountVectorizer` incluso puede contar n-gramas. Si queremos tanto unigramas (palabras individuales) como bigramas, especificamos: `ngram_range=(1, 2)`

Si queremos solo bigramas, usamos: `ngram_range=(2, 2)`

Ahora, vamos a obtener unigramas, bigramas y trigramas para nuestros documentos.

In [ ]:
vec = CountVectorizer(ngram_range=(1, 3), lowercase=True)
# Eliminar las dos primeras líneas de cada documento
docs_borges_noheader = docs_borges.apply(lambda text: "\n".join(text.splitlines()[2:]))
vec.fit(docs_borges_noheader)
vec.transform(docs_borges_noheader)

In [ ]:
# numero de valores que no son cero en la matriz dispersa.
vec.transform(docs_borges_noheader).count_nonzero()

Hay casi 11,000 bigramas.
Si quisiéramos almacenar estos datos en un `DataFrame`, necesitaríamos tantas columnas, aunque solo aproximadamente 12,000 de las casi 66,000 (6 * 11,000 bigramas) entradas sean distintas de cero.

Por eso, las matrices dispersas (_sparse matrices_) son vitales en el procesamiento de texto, ya que permiten almacenar y manipular estos datos de manera eficiente sin desperdiciar memoria en los ceros.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(ngram_range=(2, 2), lowercase=True)  # solo bigramas
tf_sparse = vec.fit_transform(docs_borges_noheader)
print(
    len(vec.get_feature_names_out())
)  # Esto devuelve el número de columnas, es decir, el número de bigramas
print(tf_sparse.nnz)  # Número de entradas no nulas
tf_sparse.shape[0] * tf_sparse.shape[1]  # Número total de entradas

## TF-IDF

El problema con las frecuencias de términos (TF) es que las palabras comunes como "el" y "que" tienden a tener conteos altos y dominan la matriz. Un mejor indicador de si dos documentos son similares es si comparten palabras raras.

Por ejemplo, la palabra "evangelio" solo aparece en dos de los cuentos de Borges. Que esa palabra esté presente en dos documentos es un indicador fuerte de similitud, por lo que deberíamos dar más peso a términos como este.

Esta es la idea detrás de **TF-IDF**. Tomamos cada **frecuencia de término** (TF) y la reajustamos según en cuántos documentos aparece ese término (es decir, la **frecuencia de documentos**, DF). Como queremos que las palabras que aparecen en menos documentos tengan más peso, usamos la **frecuencia inversa de documentos** (IDF).
Tomamos el logaritmo del IDF porque la distribución de los IDF está fuertemente sesgada hacia la derecha.

Al final, la fórmula del IDF es:

$$ \textrm{idf}(t, D) = \log \frac{\text{# de documentos}}{\text{# de documentos que contienen $t$}} = \log \frac{|D|}{|d \in D: t \in d|}. $$

(A veces se añade un $1$ en el denominador para evitar la división por cero, si hay términos en el vocabulario que no aparecen en el corpus). Para calcular TF-IDF, simplemente multiplicamos las frecuencias de términos por las frecuencias inversas de documentos:

$$ \textrm{tf-idf}(d, t, D) = \textrm{tf}(d, t) \cdot \textrm{idf}(t, D). $$

Fíjate que, a diferencia de TF, la representación TF-IDF de un documento dado depende de todo el corpus de documentos.

Primero veamos cómo calcular TF-IDF desde cero, usando la matriz de frecuencia de términos que obtuvimos anteriormente.

In [ ]:
# Obtener las frecuencias de los documentos
# En cuantos documentos aparece cada término?)
df = (tf > 0).sum(axis=0)
df

In [ ]:
import numpy as np

# Obtener IDFs
idf = np.log(len(tf) / df)
idf

In [ ]:
# Calcular TF-IDFs
tf_idf = tf * idf
tf_idf

### Implementación usando scikit-learn

Generalmente no implementaremos TF-IDF desde cero, como hicimos antes.
En su lugar, usaremos el `TfidfVectorizer` de `Scikit-Learn`, que funciona de manera similar a `CountVectorizer`, excepto que devuelve una matriz de pesos TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(norm=None, lowercase=True)  # No normalizar.
# Eliminar las dos primeras líneas de cada documento
docs_borges_noheader = docs_borges.apply(lambda text: "\n".join(text.splitlines()[2:]))
vec.fit(docs_borges_noheader)  # Esto determina el vocabulario
tf_idf_sparse = vec.transform(docs_borges_noheader)
tf_idf_sparse

## Similitud del coseno

Ahora tenemos una representación de cada documento de texto como un vector de números. Cada número puede ser una frecuencia de término (TF) o un peso TF-IDF. Podemos visualizar cada vector como una flecha en un espacio de alta dimensión, donde cada dimensión representa una palabra. La magnitud del vector en una dimensión representa la "frecuencia" (TF o TF-IDF) de esa palabra en el documento.

Por ejemplo, si nuestro vocabulario solo contiene dos palabras, "yo" y "evangelio", entonces las flechas podrían representar dos documentos:



In [ ]:
from IPython.display import Image, display

display(Image(filename=os.path.join(currentDirectory, "img", "vector_space.png"), width=300))

Necesitamos una forma de medir la distancia entre dos documentos (es decir, entre dos vectores). Podríamos usar la distancia Euclidiana, como se ha hecho antes:

In [ ]:
from IPython.display import Image, display

display(Image(filename=os.path.join(currentDirectory, "img", "vector_space_euclidean.png"), width=300))

Por qué la distancia Euclidiana no es adecuada

Considera estos dos documentos:

"Yo soy Sam."

"Yo soy Sam. Sam soy yo."

Los dos documentos son claramente muy similares, pero el vector del segundo es el doble de largo que el del primero porque tiene el doble de ocurrencias de cada palabra. Esto ocurre tanto con TF como con TF-IDF.

Si calculamos la distancia Euclidiana entre estos vectores, parecerán lejos, aunque son casi idénticos en contenido.

In [ ]:
from IPython.display import Image, display

display(Image(filename=os.path.join(currentDirectory, "img", "vector_space_example.png"), width=300))

Con vectores TF o TF-IDF, la propiedad distintiva es su dirección. Como los dos vectores anteriores apuntan en la misma dirección, son similares.

Necesitamos una métrica que mida qué tan diferentes son sus direcciones. Una forma natural de medir la diferencia entre dos vectores es el ángulo entre ellos:

In [ ]:
from IPython.display import Image, display

display(Image(filename=os.path.join(currentDirectory, "img", "vector_space_cosine.png"), width=300))

El coseno del ángulo entre dos vectores $a$ y $b$ se calcula como:

$$ \cos \theta = \frac{\sum a_j b_j}{\sqrt{\sum a_j^2} \sqrt{\sum b_j^2}}. $$

Aunque es posible calcular el ángulo $\theta$ a partir de esta fórmula, es más común calcular $\cos\theta$ como medida de similitud entre dos vectores.

Esta métrica de similitud se llama **similitud coseno**.
Observa que cuando el ángulo $\theta$ es cercano a 0 (es decir, cuando los dos vectores apuntan casi en la misma dirección), el valor de $\cos\theta$ es alto — cercano a 1.0, que es el valor máximo posible.

La distancia coseno se define como 1 menos la similitud.
De esta manera, un valor de 0 significa que los dos vectores apuntan exactamente en la misma dirección.

$$ d_{\cos}({\bf a}, {\bf b}) = 1 - \cos(\theta({\bf a}, {\bf b})) = 1 - \frac{\sum a_j b_j}{\sqrt{\sum a_j^2} \sqrt{\sum b_j^2}}. $$

Vamos a calcular la similitud coseno entre el documento 0 (_el_zahir_) y el documento 4 (_deutsches_requiem_) usando la representación TF-IDF.

In [ ]:
# Calcular el numerador.
a = tf_idf_sparse[0, :]
b = tf_idf_sparse[4, :]
dot = a.multiply(b).sum()

# Calcular los términos en el denominador.
a_len = np.sqrt(a.multiply(a).sum())
b_len = np.sqrt(b.multiply(b).sum())

# La similaridad del coseno se calcula como su ratio.
dot / (a_len * b_len)

Estos dos vectores son muy similares, como lo evidencia su alta similitud coseno (cercana a 1.0).
Vamos a intentar encontrar los documentos más similares del corpus a _el_zahir_ —es decir, sus vecinos más cercanos.

Para hacer esto, aprovecharemos el broadcasting: multiplicaremos un vector TF-IDF (que representa el documento 0) por toda la matriz TF-IDF y calcularemos la suma sobre las columnas. Esto nos dará un vector de productos escalar.

In [ ]:
# Calcular los numeradores.
a = tf_idf_sparse[0, :]
B = tf_idf_sparse
dot = a.multiply(B).sum(axis=1)
dot

In [ ]:
# Calcular los denominadores.
a_len = np.sqrt(a.multiply(a).sum())
b_len = np.sqrt(B.multiply(B).sum(axis=1))
print(a_len)
b_len

In [ ]:
# Calcular sus ratios para obtener las similaridades del coseno.
dot / (a_len * b_len)

Ahora vamos a poner esta matriz en un `DataFrame` para poder ordenar fácilmente los valores de forma descendente.

In [ ]:
cos_similarities = pd.DataFrame(dot / (a_len * b_len))[0]
most_similar = cos_similarities.sort_values(ascending=False)
most_similar

Por supuesto, el documento más similar en el corpus a _el_zahir_ (con una similitud coseno perfecta de 1.0) es él mismo. Pero el siguiente texto más similar es _deutsches_requiem_ seguida de _funes_el_memorioso_.

### Implementación usando scikit-learn

Para calcular la similitud coseno entre dos documentos usando la representación TF-IDF en Python con `Scikit-Learn`, puedes hacer lo siguiente:

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances

cosine_similarity(tf_idf_sparse)

La entrada $(i,j)$ de esta matriz representa la similitud coseno entre el $i$-ésimo y el $j$-ésimo documento. Por lo tanto, la primera fila de esta matriz contiene las similitudes entre _el_zahir_ y los demás documentos del corpus.
Verifica que estos números coincidan con los que obtuvimos manualmente.

In [ ]:
cosine_similarity(tf_idf_sparse)[0]

# 3️⃣ Ejemplo práctico con Whoosh

Whoosh es un motor de búsqueda que utiliza un índice invertido para almacenar los documentos. Cuando realizamos una consulta, Whoosh no recorre todos los documentos; en su lugar, utiliza este índice para localizar rápidamente los documentos que contienen los términos de la query.

Para ordenar los resultados según relevancia, Whoosh implementa varios modelos de scoring, siendo uno de los más comunes el TF-IDF (_Term Frequency – Inverse Document Frequency_).

In [ ]:
!pip install whoosh

El ranking **TF-IDF** de Whoosh se basa en dos componentes fundamentales:

- **TF (Term Frequency)**: la frecuencia del término en un documento. Cuanto más veces aparezca una palabra de la query en un documento, mayor será su contribución al score.

- **IDF (Inverse Document Frequency)**: la importancia del término en el corpus. Las palabras comunes (como “el”, “la”, “de”) aparecen en muchos documentos y tienen poco poder discriminante, por lo que su IDF es bajo. En cambio, palabras raras que aparecen en pocos documentos reciben un IDF alto.

El score TF-IDF para un término \(t\) en un documento \(d\) se calcula aproximadamente como:

$$
\text{score}_{t,d} = tf_{t,d} \times \log\frac{N}{df_t + 1}
$$

donde:

- ($N$) = número total de documentos en el índice  
- ($df_t$) = número de documentos que contienen el término ($t$)

El score total de un documento con respecto a una query es la **suma de los scores TF-IDF de todos los términos de la query**:

$$
\text{score}_d = \sum_{t \in \text{query}} \text{score}_{t,d}
$$

In [ ]:
from whoosh.index import create_in
from whoosh.fields import Schema, TEXT, ID
from whoosh.qparser import QueryParser
from whoosh.qparser import MultifieldParser, OrGroup
from whoosh import scoring
import os, shutil
from whoosh.index import exists_in

# --- 1. Definir esquema del índice ---
schema = Schema(title=TEXT(stored=True), content=TEXT(stored=True))

# --- 2. Crear el índice solo si no existe ---
if not exists_in("indexdir"):
    os.mkdir("indexdir")
    ix = create_in("indexdir", schema)
else:
    from whoosh.index import open_dir

    ix = open_dir("indexdir")

# --- 3. Agregar documentos ---
# --- Solo agregar documentos si el índice está vacío ---
with ix.searcher() as searcher:
    num_docs = searcher.doc_count_all()

if num_docs == 0:
    writer = ix.writer()
    writer.add_document(title="El Zahir", content=docs_borges_noheader.iloc[0])
    writer.add_document(
        title="evangelio_segun_marcos", content=docs_borges_noheader.iloc[1]
    )
    writer.add_document(
        title="funes_el_memorioso", content=docs_borges_noheader.iloc[2]
    )
    writer.add_document(
        title="la_biblioteca_de_babel", content=docs_borges_noheader.iloc[3]
    )
    writer.add_document(title="deutsches_requiem", content=docs_borges_noheader.iloc[4])
    writer.add_document(
        title="la_loteria_en_babilonia", content=docs_borges_noheader.iloc[5]
    )
    writer.commit()
    print("✅ Documentos indexados correctamente.")
else:
    print(f"⚠️ El índice ya contiene {num_docs} documentos. No se añadieron nuevos.")

q = "perro gato"  # "evangelio"
# --- 4. Buscar usando modelo vectorial (TF-IDF) ---
with ix.searcher(weighting=scoring.TF_IDF()) as searcher:
    query = QueryParser("content", ix.schema, group=OrGroup).parse(q)
    results = searcher.search(query, limit=None)

    print("\n🔹 Resultados con modelo TF-IDF:\n")
    for hit in results:
        print(f"Documento: {hit['title']}, Puntuación: {hit.score:.4f}")

**¿Qué hace este ejemplo?**

Usando la consulta "*perro gato*" usa el modelo TF_IDF para calcular la similitud entre la consulta y los documentos indexados.

Al final suma los scores.


### Limitaciones de Whoosh y de sumar los scores TF-IDF

En la explicación anterior, mencionamos que el **score total de un documento** se puede calcular sumando los scores TF-IDF de todos los términos de la query:

$$
\text{score}_d = \sum_{t \in \text{query}} \text{score}_{t,d}
$$

Si bien esta suma es un enfoque sencillo, **no captura la similitud real entre documentos**, por varias razones:

1. **No considera la longitud del documento**: un documento más largo tendrá más términos y, por lo tanto, tiende a tener un score más alto, aunque su contenido no sea más relevante.
2. **No considera la dirección del vector de términos**: TF-IDF convierte cada documento en un vector de pesos, y la suma ignora cómo se distribuyen estos pesos entre los términos.
3. **No normaliza los efectos de frecuencia alta**: un término muy frecuente en un documento puede dominar la suma y sesgar el ranking.

> **Ejemplo:** probar la consulta `"evangelio"` en el ejercicio anterior y observar los resultados.
>
> * En Whoosh, aunque los documentos tengan frecuencias diferentes (por ejemplo 4 y 3), el score final TF-IDF depende de la fórmula interna que combina TF e IDF, además de otros factores como normalización de campo y peso del documento.
>
> * Es posible que las diferencias pequeñas en frecuencia no se reflejen mucho en el ranking si el IDF del término es bajo o si el documento es largo. Esto puede dar la impresión de que el score es muy parecido o casi igual, aunque no sea exactamente el mismo.

### Limitaciones de Whoosh con TF-IDF

Aunque Whoosh permite rankear documentos usando TF-IDF mediante su **índice invertido**, **no calcula la similitud coseno** entre la query y los documentos.  

- Whoosh calcula el **score de cada documento** sumando la contribución de cada término de la query según TF y IDF, como vimos antes:

$$
\text{score}_d = \sum_{t \in \text{query}} \text{score}_{t,d}
$$

- Esto tiene las limitaciones mencionadas anteriormente: no normaliza por la longitud del documento ni considera la dirección de los vectores de términos, lo que puede sesgar el ranking.

### Solución: similitud coseno

Una forma más precisa de medir la relevancia es **tratar cada documento y la query como vectores TF-IDF en un espacio multidimensional** y medir el **ángulo entre ellos**:

$$
\cos \theta = \frac{\vec{d} \cdot \vec{q}}{\|\vec{d}\| \|\vec{q}\|}
$$

- $\vec{d}$ = vector TF-IDF del documento  
- $\vec{q}$ = vector TF-IDF de la query  

La similitud coseno **normaliza por la longitud del documento** y considera la distribución de los pesos de los términos, proporcionando un ranking más confiable.



### Implementación práctica usando Whoosh + Scikit-Learn

Si queremos usar la similitud coseno, necesitamos **extraer los documentos de Whoosh** y calcular la similitud con Scikit-Learn:



In [ ]:
import re


def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)  # quitar puntuación
    return text


docs_processed = docs_borges_noheader.apply(preprocess)

from whoosh.fields import Schema, TEXT, ID
from whoosh.index import create_in
import os, shutil

schema = Schema(title=ID(stored=True), content=TEXT(stored=True))

if os.path.exists("indexdir2"):
    shutil.rmtree("indexdir2")
os.mkdir("indexdir2")
ix = create_in("indexdir2", schema)

writer = ix.writer()
writer.add_document(title="El Zahir", content=docs_processed.iloc[0])
writer.add_document(title="evangelio_segun_marcos", content=docs_processed.iloc[1])
writer.add_document(title="funes_el_memorioso", content=docs_processed.iloc[2])
writer.add_document(title="la_biblioteca_de_babel", content=docs_processed.iloc[3])
writer.add_document(title="deutsches_requiem", content=docs_processed.iloc[4])
writer.add_document(title="la_loteria_en_babilonia", content=docs_processed.iloc[5])
writer.commit()

In [ ]:
from whoosh.qparser import QueryParser, OrGroup
from whoosh import scoring
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

q = "evangelio"
# 1. Abrir índice y searcher
with ix.searcher(weighting=scoring.TF_IDF()) as searcher:
    # 2. Buscar todos los documentos que contengan los términos de la query
    query = QueryParser("content", ix.schema, group=OrGroup).parse(q)
    results = searcher.search(query, limit=None)  # limit=None devuelve todos

    # 3. Extraer contenido y títulos
    docs = [hit["content"] for hit in results]
    titles = [hit["title"] for hit in results]

    # 4. Calcular TF-IDF con Scikit-Learn
    vectorizer = TfidfVectorizer()
    tfidf_matrix_sparse = vectorizer.fit_transform(docs)

    query_vec = vectorizer.transform([q])

    # 5. Similitud coseno
    sims = cosine_similarity(query_vec, tfidf_matrix_sparse).flatten()

    # 6. Ordenar y mostrar ranking
    sorted_idx = np.argsort(sims)[::-1]
    for i in sorted_idx:
        print(f"{titles[i]}: {sims[i]:.5f}")

**¿Cuál es el problema de esta solución?**

Lo que está pasando es lo siguiente:

1. `vectorizer.fit_transform(docs)` crea el vocabulario y calcula los pesos TF-IDF solo a partir de los documentos que devolvió Whoosh para la query "evangelio".

2. Esto significa que los documentos que no contienen “evangelio” no se incluyen en el cálculo de IDF.

3. Como consecuencia, las palabras raras o comunes se ponderan solo según los documentos filtrados, no según el corpus completo.

**Efectos:**

* Los TF-IDF pueden ser más altos o más bajos de lo que serían si calcularas el IDF sobre todo el corpus.

* Las similitudes coseno se comparan correctamente solo entre los documentos devueltos, pero no son comparables con documentos que no aparecen en los resultados, porque su IDF no está considerada.

* Si el objetivo es hacer un ranking consistente sobre todo el corpus, necesitas calcular la matriz TF-IDF sobre todos los documentos primero, y luego transformar la query y los documentos relevantes para calcular similitud coseno.

# 4️⃣ Implementación de un indexador mediante índices invertidos y modelo VSM

In [ ]:
from collections import defaultdict
from functools import reduce
import math
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. Diccionario de documentos ---
document_filenames = {i: os.path.join(borges_dir, fname) for i, fname in enumerate(borges_files)}

# --- 2. Inicializar postings e índice invertido ---
postings = defaultdict(set)  # término -> conjunto de ids de documentos
docs_text = {}  # id -> texto completo del documento
characters = ' .,!#$%^&*();:\n\t\\"?!{}[]<>'


def tokenize(text):
    terms = text.lower().split()
    return [t.strip(characters) for t in terms]


for id, path in document_filenames.items():
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    # Ignorar las dos primeras líneas
    text = "\n".join(text.splitlines()[2:])
    docs_text[id] = text
    terms = set(tokenize(text))
    for term in terms:
        postings[term].add(id)


def union(sets):
    return reduce(set.union, sets)


# --- 3. Búsqueda híbrida ---
def search_hybrid(query):
    query_terms = tokenize(query)

    # 3a. Filtrar documentos que contienen al menos un término de la query
    relevant_ids = union([postings[term] for term in query_terms if term in postings])
    if not relevant_ids:
        print("Ningún documento contiene los términos de la consulta.")
        return

    # 3b. Construir lista de documentos filtrados
    filtered_docs = [docs_text[id] for id in relevant_ids]
    filtered_ids = list(relevant_ids)

    # 3c. Calcular TF-IDF solo sobre documentos filtrados
    vectorizer = TfidfVectorizer(lowercase=True)
    tfidf_matrix = vectorizer.fit_transform(filtered_docs)

    # Transformar la query
    query_vec = vectorizer.transform([query])

    # 3d. Calcular similitud coseno
    sims = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Ordenar y mostrar resultados
    sorted_idx = sims.argsort()[::-1]
    print("Score: nombre del archivo")
    for i in sorted_idx:
        doc_id = filtered_ids[i]
        print(f"{sims[i]:.5f}: {document_filenames[doc_id]}")


# --- 4. Ejecución ---
if __name__ == "__main__":
    while True:
        q = input("Consulta >> ")
        if not q.strip():
            break
        search_hybrid(q)